# Stage 9 — Feature Priority Scoring

Generate an **actionable ranked priority list** of game features that developers should
focus on improving.

**Priority Score Formula:**
```
Priority Score = log(Total Mentions + 1) × Negative Ratio × Helpfulness Weight
```

Where:
- `Total Mentions` = number of reviews discussing this feature
- `Negative Ratio` = proportion of negative VADER sentiment reviews
- `Helpfulness Weight` = 1 + log(avg_helpful_votes + 1) (upweights features discussed by helpful reviewers)

**Priority Tiers:**
- 🔴 High — Top 33%
- 🟡 Medium — Middle 33%
- 🟢 Low — Bottom 33%

**Output:** `data/processed/feature_priority_report.csv`

## 1. Imports & Data Load

In [3]:
import pandas as pd
import numpy as np
import json
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

pd.set_option('display.max_colwidth', 120)
sns.set_theme(style='darkgrid')

sentiment_df = pd.read_csv('../data/processed/sentiment_by_feature.csv')
print(f'Loaded sentiment data for {len(sentiment_df)} categories')
print(sentiment_df.to_string(index=False))

FileNotFoundError: [Errno 2] No such file or directory: '../data/processed/sentiment_by_feature.csv'

## 2. Compute Priority Score

In [ ]:
df = sentiment_df.copy()

# Components
df['frequency_weight'] = np.log1p(df['total_mentions'])
df['neg_ratio'] = df['neg_ratio'].clip(0.01, 1.0)  # avoid zero
df['helpfulness_weight'] = 1 + np.log1p(df['avg_helpful_votes'])

# Priority Score
df['priority_score'] = (
    df['frequency_weight'] *
    df['neg_ratio'] *
    df['helpfulness_weight']
)

# Normalize to 0–100 scale
df['priority_score_normalized'] = (
    (df['priority_score'] - df['priority_score'].min()) /
    (df['priority_score'].max() - df['priority_score'].min())
) * 100

df = df.sort_values('priority_score', ascending=False).reset_index(drop=True)

print('Priority scores computed.')
print(df[['category', 'total_mentions', 'neg_ratio', 'priority_score', 'priority_score_normalized']].to_string(index=False))

In [ ]:
# Assign tiers based on priority score
thresholds = df['priority_score'].quantile([0.33, 0.67])
low_thresh = thresholds[0.33]
high_thresh = thresholds[0.67]

def assign_tier(score):
    if score >= high_thresh:
        return '🔴 High'
    elif score >= low_thresh:
        return '🟡 Medium'
    else:
        return '🟢 Low'

def assign_tier_label(score):
    if score >= high_thresh:
        return 'High'
    elif score >= low_thresh:
        return 'Medium'
    else:
        return 'Low'

df['priority_tier'] = df['priority_score'].apply(assign_tier)
df['tier_label'] = df['priority_score'].apply(assign_tier_label)

print(f'Tier thresholds: Low < {low_thresh:.3f} ≤ Medium < {high_thresh:.3f} ≤ High')
print(df['priority_tier'].value_counts())

## 3. Final Priority Report

In [ ]:
report = df[[
    'category', 'total_mentions', 'positive', 'negative', 'neutral',
    'pos_ratio', 'neg_ratio', 'avg_compound',
    'avg_helpful_votes', 'priority_score', 'priority_score_normalized',
    'priority_tier', 'tier_label'
]].copy()

# Pretty display table
display_cols = ['category', 'total_mentions', 'positive', 'negative', 'neg_ratio', 'priority_score_normalized', 'priority_tier']
display_df = report[display_cols].copy()
display_df.columns = ['Feature', 'Mentions', 'Positive', 'Negative', 'Neg. Ratio', 'Priority Score', 'Priority']
display_df['Neg. Ratio'] = display_df['Neg. Ratio'].apply(lambda x: f'{x:.1%}')
display_df['Priority Score'] = display_df['Priority Score'].apply(lambda x: f'{x:.1f}/100')
display_df['Feature'] = display_df['Feature'].str.replace('_', '/')

print('=' * 80)
print('GAME FEATURE PRIORITY REPORT — Cyberpunk 2077 Steam Reviews')
print('=' * 80)
print(display_df.to_string(index=False))
print('=' * 80)
print()
print('Legend: 🔴 High Priority = Fix urgently | 🟡 Medium = Improve | 🟢 Low = Maintain')

## 4. Priority Score Visualization

In [ ]:
fig, ax = plt.subplots(figsize=(13, 7))

tier_colors = {'High': '#e74c3c', 'Medium': '#f39c12', 'Low': '#2ecc71'}
colors = [tier_colors[t] for t in df['tier_label']]

bars = ax.barh(
    df['category'].str.replace('_', '/'),
    df['priority_score_normalized'],
    color=colors, alpha=0.9, edgecolor='white', linewidth=0.5
)

ax.bar_label(bars, labels=[f'{v:.1f}' for v in df['priority_score_normalized']], padding=3, fontsize=9)
ax.set_xlabel('Priority Score (0–100)', fontsize=12)
ax.set_title('Feature Priority Score — Cyberpunk 2077 Steam Reviews\n(Higher = Needs More Developer Attention)', fontsize=13, fontweight='bold')
ax.invert_yaxis()

legend_patches = [
    mpatches.Patch(color='#e74c3c', label='🔴 High Priority'),
    mpatches.Patch(color='#f39c12', label='🟡 Medium Priority'),
    mpatches.Patch(color='#2ecc71', label='🟢 Low Priority'),
]
ax.legend(handles=legend_patches, loc='lower right', fontsize=10)
ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('../data/processed/feature_priority_chart.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved chart.')

In [ ]:
# Bubble chart: Mentions vs Sentiment vs Priority
fig, ax = plt.subplots(figsize=(13, 8))

scatter = ax.scatter(
    df['total_mentions'],
    df['neg_ratio'],
    s=df['priority_score_normalized'] * 25,
    c=[{'High': '#e74c3c', 'Medium': '#f39c12', 'Low': '#2ecc71'}[t] for t in df['tier_label']],
    alpha=0.8, edgecolors='white', linewidth=1.5
)

# Annotate categories
for _, row in df.iterrows():
    ax.annotate(
        row['category'].replace('_', '/'),
        (row['total_mentions'], row['neg_ratio']),
        textcoords='offset points', xytext=(8, 4), fontsize=9
    )

ax.set_xlabel('Total Mentions (Review Volume)', fontsize=12)
ax.set_ylabel('Negative Sentiment Ratio', fontsize=12)
ax.set_title('Feature Landscape: Volume × Negative Sentiment\n(Bubble size = Priority Score)', fontsize=13, fontweight='bold')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))

ax.legend(handles=legend_patches, loc='upper left', fontsize=10)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('../data/processed/feature_bubble_chart.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved chart.')

## 5. Save Results

In [ ]:
report.to_csv('../data/processed/feature_priority_report.csv', index=False)
print(f'Saved feature_priority_report.csv')

# Also save as JSON for dashboard
report_dict = report.to_dict(orient='records')
with open('../data/processed/feature_priority_report.json', 'w') as f:
    json.dump(report_dict, f, indent=2)
print('Saved feature_priority_report.json')

print('\n✅ Stage 9 Complete — Feature Priority Scoring Done!')
print(f'\nTop 3 Features to Fix:')
for _, row in report.head(3).iterrows():
    print(f"  {row['priority_tier']} {row['category'].replace('_', '/')} — {row['total_mentions']:,} mentions, {row['neg_ratio']:.1%} negative")